## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 47 samples and 12497 columns.
First few rows of the data:


,efstime,efscens,stime,scens,sample_title,tissue,age at_diag,age at_first_event_or_censor,ethnicity,SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,4043,0,4043,0,Neuroblastoma_usi_PALUKI,Neuroblastoma,1110,5152,Not Hispanic or Latino,1.747648,...,7.059210,4.891928,6.500045,8.330441,5.795434,5.946433,7.427408,6.659727,4.882422,6.943524
1,245,0,245,0,Neuroblastoma_usi_PALWKV,Neuroblastoma,2019,2263,Not Hispanic or Latino,2.440854,...,6.377251,4.579810,6.232081,7.965482,5.318799,5.543398,7.070861,6.233269,4.754535,6.328568
2,206,1,206,1,Neuroblastoma_usi_PAMKGY,Neuroblastoma,926,1131,Not Hispanic or Latino,2.622430,...,5.618590,3.651019,5.525917,7.109841,4.350829,4.528510,6.063517,5.487258,3.774227,5.495584
3,2466,0,2466,0,Neuroblastoma_usi_PAMMGU,Neuroblastoma,953,3418,Hispanic or Latino,1.796477,...,6.366620,4.494385,6.178376,7.621467,5.154473,5.125012,6.773641,6.126936,4.257590,6.334392
4,3838,0,3838,0,Neuroblastoma_usi_PANLCE,Neuroblastoma,1960,5797,Not Hispanic or Latino,2.633410,...,5.879444,4.100270,5.663046,6.738357,4.862939,5.308358,6.844380,6.236861,4.448032,5.939340


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
